# BM4Implicit versus RK4 — calculation

Run this notebook with the project `.venv` kernel to compute and persist all three particle studies. Then open [visualization.ipynb](visualization.ipynb) to load the saved results and generate figures without integrating again.

Three independent guiding centers start at radial fractions (0.1, 0.2, 0.3), angle zero, in the PHI_2.h5 field. This notebook runs the existing planar energy study separately for each saved particle, through **normalized time 35** (the reference time axis, not a separately converted cycle count).

Run with the project `.venv` kernel. The checksummed standard reference in `data/trajectory/h5_three_radial_dop853_t35` is loaded directly; DOP853 and Radau are never recomputed. BM4/RK4 runs retain three fixed step sizes and three alternating timing repetitions, so the full comparison can take substantial time.

`DISPLAY_PARTICLE` selects the particle used for printed diagnostics. The two-method scope is deliberate; each particle retains the existing planar structural and energy diagnostics.

## 1. Physical energy is not an invariant of this time-dependent field

The project convention is

$$H(t,x,y)=\langle\Phi\rangle_\rho(t,x,y),\qquad
\dot x=-\partial_y H,\quad \dot y=\partial_x H.$$

With `indx=(0,1)`, the loaded potential includes the mean and the dominant
positive-frequency mode. Consequently,

$$\frac{dH(t,z(t))}{dt}=\partial_tH(t,z(t)),$$

so **$H(t,z_n)-H(0,z_0)$ is not a conservation error**. We distinguish:

| Quantity | Definition | Interpretation |
|---|---|---|
| Physical variation | $H(t,z_n)-H(0,z_0)$ | Includes true time dependence |
| Physical-energy error | $H(t,z_n)-H(t,z_{\rm DOP853})$ | Accuracy against the same interpolated ODE |
| Extended balance | $\kappa'=-\partial_tH$, $K=H+\kappa$, $\kappa(0)=0$ | $K$ is constant in the continuous augmented system |
| Envelope | $E_K(T,h)=\max_{0\le t_n\le T}|K_n-K_0|$ | Does the largest error continue to grow? |
| Trajectory error | minimum-image distance to DOP853 | Energy accuracy does not establish orbit accuracy |

BM4's $\kappa$ is reconstructed by `GCGeneralizedEnergyObserver` from its twelve
accepted base stages, with the project's $\kappa=k/2$ normalization. No sparse-output
quadrature is used. RK4 advances $\kappa$ with the same four RK stages as its state.

`BM4Implicit` uses one reduced Hairer projection around the complete BM4 composition.
It accepts the **physical state only**; the observer does not turn it into a fully
extended solver. A small physical symplectic defect at fixed time does not prove
that the reconstructed $(x,y,t,\kappa)$ map is symplectic.

The familiar $O(h^4)$ long-time energy estimate requires a suitably regular
autonomous Hamiltonian, a symplectic map, a small constant step, and a bounded
regularity domain. Those hypotheses must not be assumed from a finite plot.
In particular, cubic spatial splines have limited regularity, and Newton error,
roundoff, reference resolution and chaotic separation can influence the measured
envelopes. The $h^4$ lines below are comparison guides, not guaranteed bounds.

In [1]:
from pathlib import Path
from dataclasses import asdict
from datetime import datetime
from uuid import uuid4
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from diagnostics.paths import find_project_root, notebook_output_directory
from diagnostics.reference_trajectory import load_reference_trajectory
from diagnostics.gc_energy_bound import save_energy_bound_result, load_energy_bound_result
from potential import load_gc2d_h5_potential
from dynamics import GuidingCenterDynamics
from studies.initial_conditions import radial_gc_configuration, domain_center
from studies.saved_radial_reference import select_reference_particle
from initial_conditions import GCInitialConfiguration
from studies.gc_energy_bound import (
    GCEnergyBoundConfig, run_gc_energy_bound_study, audit_initial_step_geometry,
    energy_bound_conclusions,
)
from visualization.gc_energy_bound import show_energy_table

ROOT = find_project_root(Path.cwd())
NOTEBOOK_PATH = ROOT / "notebooks/developements/energy/BM4_RK4/calculation.ipynb"
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
OUTPUT_DIRECTORY = notebook_output_directory(NOTEBOOK_PATH, project_root=ROOT) / RUN_TAG
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", sys.executable)
print("Project:", ROOT)
print("Run output:", OUTPUT_DIRECTORY)

Python: /home/juan/Proyectos/GC2D_intranet/.venv/bin/python
Project: /home/juan/Proyectos/GC2D_intranet
Run output: /home/juan/Proyectos/GC2D_intranet/outputs/developements/energy/BM4_RK4/calculation/2026-09-14/20260914_115553_d9871731


## 2. Explicit and editable experiment parameters

One unit of normalized time is the selected characteristic period $T_0=2\pi/\omega_0$.
The physical coordinate scale is $\lambda=0.06$ and
$\hat x=2\pi(R-R_0)/\lambda$. The potential normalization is
$\hat\Phi=(2\pi)^2\Phi/(\omega_0\lambda^2B)$.
Actual dimensional conversion factors are printed from HDF5 metadata below.
The initial spatial radius fraction is distinct from the normalized gyro-radius `RHO`.
There is no random sampling and therefore no seed.

Keep the duration and every horizon divisible by every step, and each step an integer
multiple of the finest step. Reference errors use only times shared with the saved reference. Reducing the
step never changes it during one trajectory and never introduces a shortened final step.

For a first short run, use `T_END=2.0` and `HORIZONS=(0.5, 1.0, 2.0)`.
A stationary HDF5 control requires a separate matching reference.

The standard reference is fixed to the recorded field and initial states. Changing physical parameters requires a different audited reference. Steps are (0.1, 0.05, 0.025). The stored reference spacing is 0.01. For h=0.025, reference comparisons use every second accepted node (spacing 0.05); all 1401 method states and energy-balance values are retained. For h=0.1 and h=0.05, every accepted node is shared. No interpolation is used.

In [2]:
# Measured field: preserve the native spatial resolution and use cubic splines.
H5_PATH = ROOT / "data/potential/V1/PHI_2.h5"
MAGNETIC_FIELD = 1.5
CHARACTERISTIC_LENGTH = 0.06
FIELD_INDICES = (0, 1)
INTERPOLATION_ORDER = 3
RHO = 0.3
RADIAL_FRACTIONS = (0.1, 0.2, 0.3)  # Saved reference geometry.
INITIAL_ANGLE = 0.0            # Radians from +x.

T_START, T_END = 0.0, 35.0
STEPS = (0.1, 0.05, 0.025)
HORIZONS = (1.0, 5.0, 10.0, 25.0, 35.0)
COUPLING_FREQUENCY = float(np.pi / 8)
NEWTON_ATOL, NEWTON_RTOL = 1e-12, 1e-11
NEWTON_MAX_ITERATIONS = 40
JACOBIAN_RELATIVE_STEP = float(np.cbrt(np.finfo(float).eps))
TIMING_REPEATS = 3
BLOCK_COUNT = 10
PLATEAU_RELATIVE_GROWTH = 0.10  # Descriptive last-window criterion; never a theorem.

REFERENCE_DIRECTORY = ROOT / "data/trajectory/h5_three_radial_dop853_t35"
DISPLAY_PARTICLE = 0  # Zero-based particle index for detailed figures: 0, 1 or 2.
DISPLAY_LEVEL = 0           # 0 uses the coarsest step; change to inspect another refinement.
ANIMATION_FRAMES, ANIMATION_FPS = 201, 10

config = GCEnergyBoundConfig(
    t_span=(T_START, T_END), steps=STEPS, horizons=HORIZONS, rho=RHO,
    coupling_frequency=COUPLING_FREQUENCY, newton_atol=NEWTON_ATOL,
    newton_rtol=NEWTON_RTOL, newton_max_iterations=NEWTON_MAX_ITERATIONS,
    jacobian_relative_step=JACOBIAN_RELATIVE_STEP, timing_repeats=TIMING_REPEATS,
    block_count=BLOCK_COUNT, plateau_relative_growth=PLATEAU_RELATIVE_GROWTH,
)
assert H5_PATH.is_file(), f"Missing measured HDF5 potential: {H5_PATH}"
assert 0 <= DISPLAY_LEVEL < len(STEPS)
print("Fixed steps per method:", [round((T_END-T_START)/h) for h in STEPS])
assert 0 <= DISPLAY_PARTICLE < 3
standard_reference = load_reference_trajectory(REFERENCE_DIRECTORY)
assert standard_reference.times[0] == T_START and standard_reference.times[-1] == T_END
print("Reference saved states:", standard_reference.times.size)

Fixed steps per method: [350, 700, 1400]
Reference saved states: 3501


In [3]:
potential = load_gc2d_h5_potential(
    H5_PATH, B=MAGNETIC_FIELD, characteristic_length=CHARACTERISTIC_LENGTH,
    indx=FIELD_INDICES, interpolation_order=INTERPOLATION_ORDER,
    nx=None, ny=None, denoising=False, spatial_normalization="characteristic_length",
)
expected_configuration = radial_gc_configuration(
    potential, radial_fractions=RADIAL_FRACTIONS, angle=INITIAL_ANGLE)
np.testing.assert_array_equal(expected_configuration.initial_state, standard_reference.initial_state)
references = [select_reference_particle(standard_reference, i) for i in range(3)]
configurations = [GCInitialConfiguration.from_components(x=ref.initial_state[:1], y=ref.initial_state[1:])
                  for ref in references]
dynamics = GuidingCenterDynamics(potential, rho=RHO)
configuration = configurations[DISPLAY_PARTICLE]
initial_state = configuration.initial_state
initial_rows = [{"particle": i+1, "x0": float(ref.initial_state[0]),
                 "y0": float(ref.initial_state[1]), "radial_fraction": RADIAL_FRACTIONS[i],
                 "angle": INITIAL_ANGLE} for i, ref in enumerate(references)]
show_energy_table(initial_rows, ("particle", "x0", "y0", "radial_fraction", "angle"))
source = potential.metadata
potential_metadata = {
    "source_path": str(H5_PATH), "B": MAGNETIC_FIELD,
    "characteristic_length": CHARACTERISTIC_LENGTH, "indx": list(FIELD_INDICES),
    "interpolation_order": INTERPOLATION_ORDER, "denoising": False,
    "spatial_normalization": "characteristic_length", "resampling": None,
    "source_field_indices": source.source_field_indices.tolist(),
    "source_frequencies": source.source_frequencies.tolist(),
    "characteristic_period": source.characteristic_period,
    "characteristic_frequency": source.characteristic_frequency,
    "normalization_factor": source.normalization_factor,
    "runtime_frequencies": potential.frequencies.tolist(),
    "grid_shape": list(potential.grid.shape), "cell_period": potential.grid.period,
}
initial_metadata = {"geometry": "three independent points on one radius from cell center",
                    "radial_fractions": RADIAL_FRACTIONS, "angle": INITIAL_ANGLE,
                    "initial_state": standard_reference.initial_state.tolist(), "random_seed": None}
print(json.dumps(potential_metadata, indent=2))
print("Initial Hamiltonian:", dynamics.hamiltonian(T_START, initial_state))
print("Initial partial_t H:", -dynamics.extended_momentum_derivative(T_START, initial_state))
print("Initial GC velocity:", dynamics.vector_field(T_START, initial_state))

particle,x0,y0,radial fraction,angle
1,1.13097e+01,9.42478e+00,1.00000e-01,0.00000e+00
2,1.31947e+01,9.42478e+00,2.00000e-01,0.00000e+00
3,1.50796e+01,9.42478e+00,3.00000e-01,0.00000e+00


{
  "source_path": "/home/juan/Proyectos/GC2D_intranet/data/potential/V1/PHI_2.h5",
  "B": 1.5,
  "characteristic_length": 0.06,
  "indx": [
    0,
    1
  ],
  "interpolation_order": 3,
  "denoising": false,
  "spatial_normalization": "characteristic_length",
  "resampling": null,
  "source_field_indices": [
    15
  ],
  "source_frequencies": [
    2404.106675108847
  ],
  "characteristic_period": 0.002613521842534343,
  "characteristic_frequency": 2404.106675108847,
  "normalization_factor": 0.3288423607980393,
  "runtime_frequencies": [
    1.0
  ],
  "grid_shape": [
    256,
    256
  ],
  "cell_period": 18.84955592153019
}
Initial Hamiltonian: [32.61239014]
Initial partial_t H: [2.49482215]
Initial GC velocity: [0.79506476 0.48111403]


## 3. Local structural check and reference resolution

The first-step Jacobian is measured with two centered finite-difference increments.
BM4 is identified through its twelve-stage accepted record and its reduced projection.
The physical defect is $\|D\Phi^T\Omega D\Phi-\Omega\|_F$; its measured value includes
finite-difference and nonlinear-solve errors. It is not an audit of the reconstructed
extended state and it is not extrapolated to the entire orbit.

The reference was computed with the existing high-precision DOP853/Radau pipeline and is now loaded from the standard saved artifact. Their discrepancy
is a **measured resolution indicator**, not a rigorous error bound. The pipeline
persists both trajectories, solver settings and the fingerprint of the actual
gyroaveraged interpolated field. A reused reference must match the field, radius,
initial state and shared comparison times. All three particles are studied independently.

In [4]:
geometry_audits = [audit_initial_step_geometry(potential, cfg, config=config) for cfg in configurations]
geometry_audit = geometry_audits[DISPLAY_PARTICLE]
for i, rows in enumerate(geometry_audits):
    print("Particle", i+1)
    show_energy_table(rows, ("method", "step", "fd_relative_step", "determinant", "symplectic_defect"))

Particle 1


method,step,fd relative step,determinant,symplectic defect
BM4Implicit,1.00000e-01,6.05545e-06,1.00000e+00,8.29671e-09
BM4Implicit,1.00000e-01,3.02773e-06,1.00000e+00,1.91480e-09
RK4,1.00000e-01,6.05545e-06,9.99892e-01,1.52534e-04
RK4,1.00000e-01,3.02773e-06,9.99892e-01,1.52556e-04


Particle 2


method,step,fd relative step,determinant,symplectic defect
BM4Implicit,1.00000e-01,6.05545e-06,1.00000e+00,5.31781e-09
BM4Implicit,1.00000e-01,3.02773e-06,1.00000e+00,1.41132e-09
RK4,1.00000e-01,6.05545e-06,9.95816e-01,5.91679e-03
RK4,1.00000e-01,3.02773e-06,9.95816e-01,5.91687e-03


Particle 3


method,step,fd relative step,determinant,symplectic defect
BM4Implicit,1.00000e-01,6.05545e-06,1.00000e+00,1.24142e-07
BM4Implicit,1.00000e-01,3.02773e-06,1.00000e+00,3.11209e-08
RK4,1.00000e-01,6.05545e-06,9.95866e-01,5.84577e-03
RK4,1.00000e-01,3.02773e-06,9.95865e-01,5.84807e-03


## 4. Fixed-step comparison and persistence

Each of the three saved physical orbits is integrated independently by BM4 and RK4 on each grid.
Every accepted step is saved, so extended-balance envelopes and projection maxima include all nodes. Physical-energy and trajectory errors against DOP853 use only shared reference nodes.
Reference samples are selected at coincident saved times without interpolating
the numerical method or restarting it at the intermediate horizons.

The three timing repetitions alternate method order and use one BLAS thread.
Reference construction and energy observers are excluded. An untimed replay obtains
the energy histories and must reproduce each timed physical trajectory. BM4's one
nonlinear solve per complete step and its residual acceptance are checked explicitly.
RK4 is excluded from nonlinear-work statistics.

The output HDF5 contains the reference, Radau audit, all numerical states and all
energy, work and projection histories. CSV tables are exported alongside it.

In [5]:
results, result_paths = [], []
for particle, (particle_configuration, particle_reference) in enumerate(zip(configurations, references)):
    print("Running BM4/RK4 for particle", particle+1, flush=True)
    particle_result = run_gc_energy_bound_study(
        potential, particle_configuration, config=config, reference=particle_reference)
    particle_result.metadata.update({"source": potential_metadata, "initial": initial_rows[particle],
        "geometry_audit": geometry_audits[particle], "source_notebook": str(NOTEBOOK_PATH)})
    particle_directory = OUTPUT_DIRECTORY / f"particle_{particle+1}"
    particle_directory.mkdir(parents=True, exist_ok=True)
    result_paths.append(save_energy_bound_result(particle_directory / "gc_h5_energy_bound.h5", particle_result))
    results.append(particle_result)
    print("Saved:", result_paths[-1])
result = results[DISPLAY_PARTICLE]
RESULT_PATH = result_paths[DISPLAY_PARTICLE]
print("Detailed figures show particle", DISPLAY_PARTICLE+1)

show_energy_table(result.summary, ("method", "step", "trajectory_rms", "trajectory_final",
    "H_error_rms", "H_error_max", "K_error_max", "reference_H_floor", "reference_distance_floor"))

Running BM4/RK4 for particle 1
h=0.1: 350 steps, 3 alternating timing repeats.
  repeat 1: BM4Implicit 48.39 s
  repeat 1: RK4 1.16 s
  repeat 2: RK4 1.53 s
  repeat 2: BM4Implicit 48.11 s
  repeat 3: BM4Implicit 48.28 s
  repeat 3: RK4 2.96 s
h=0.05: 700 steps, 3 alternating timing repeats.
  repeat 1: BM4Implicit 84.56 s
  repeat 1: RK4 2.35 s
  repeat 2: RK4 2.14 s
  repeat 2: BM4Implicit 80.45 s
  repeat 3: BM4Implicit 83.99 s
  repeat 3: RK4 2.58 s
h=0.025: 1400 steps, 3 alternating timing repeats.
  repeat 1: BM4Implicit 158.07 s
  repeat 1: RK4 3.70 s
  repeat 2: RK4 3.81 s
  repeat 2: BM4Implicit 153.24 s
  repeat 3: BM4Implicit 158.65 s
  repeat 3: RK4 4.39 s
Saved: /home/juan/Proyectos/GC2D_intranet/outputs/developements/energy/BM4_RK4/calculation/2026-09-14/20260914_115553_d9871731/particle_1/gc_h5_energy_bound.h5
Running BM4/RK4 for particle 2
h=0.1: 350 steps, 3 alternating timing repeats.
  repeat 1: BM4Implicit 70.91 s
  repeat 1: RK4 1.15 s
  repeat 2: RK4 1.27 s
  repe

method,step,trajectory rms,trajectory final,H error rms,H error max,K error max,reference H floor,reference distance floor
BM4Implicit,1.00000e-01,2.19097e-02,1.00422e-01,1.84743e-02,1.70616e-01,1.42052e-03,9.83631e-06,5.42480e-06
RK4,1.00000e-01,1.60232e+00,1.60754e+00,4.34905e-01,1.93936e+00,2.07967e-02,9.83631e-06,5.42480e-06
BM4Implicit,5.00000e-02,2.86154e-03,1.35202e-02,2.62545e-03,2.45964e-02,6.00999e-05,9.83631e-06,5.49732e-06
RK4,5.00000e-02,8.84140e-02,3.61446e-01,6.34255e-02,6.03457e-01,2.17132e-03,9.83631e-06,5.49732e-06
BM4Implicit,2.50000e-02,1.36281e-04,6.41898e-04,1.24612e-04,1.16406e-03,5.38169e-06,9.83631e-06,5.49732e-06
RK4,2.50000e-02,2.03739e-02,9.41147e-02,1.74317e-02,1.60854e-01,3.21943e-04,9.83631e-06,5.49732e-06


In [6]:
# Publish the completed run only after all three particle files have been saved.
manifest_path = ROOT / "outputs/developements/energy/BM4_RK4/latest_run.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest = {"run_directory": str(OUTPUT_DIRECTORY.relative_to(ROOT)),
            "reference_directory": str(REFERENCE_DIRECTORY.relative_to(ROOT))}
temporary_manifest = manifest_path.with_suffix(".tmp")
temporary_manifest.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
temporary_manifest.replace(manifest_path)
print("Completed run registered:", manifest_path)


Completed run registered: /home/juan/Proyectos/GC2D_intranet/outputs/developements/energy/BM4_RK4/latest_run.json


## Project sources and interpretation

- [HDF5 import and units](../../../../docs/dynamics/gc2d-h5-import.md).
- [Physical guiding-center dynamics](../../../../src/dynamics/gc.py).
- [BM4Implicit method and projection](../../../../docs/models/bm4-implicit/simulation/bm4-simulation-architecture.md).
- [BM4 accepted-stage energy observer](../../../../src/diagnostics/energy/observer.py).
- [Classical RK4 and optional momentum evolution](../../../../src/simulation/methods/classical/rk4.py).
- [DOP853/Radau reference pipeline](../../../../src/studies/reference_trajectory.py).
- Hairer, Lubich and Wanner, *Geometric Numerical Integration*, IX.8: the long-time
  near-conservation result is conditional; it does not assert conservation of a
  time-dependent physical Hamiltonian.

This notebook studies the same interpolated guiding-center model as the project.
It does not validate the underlying measured field or the guiding-center approximation.